# Supply Chain Domain Schema - v2 (Fixed)

This notebook creates the **supplychain** schema with the following tables:

| # | Table | Description |
|---|-------|-------------|
| 1 | `supplychain.Suppliers` | Supplier master data |
| 2 | `supplychain.ProductSuppliers` | Product-supplier mapping (many-to-many) |
| 3 | `supplychain.SupplyChainEvents` | Supply chain disruption events |
| 4 | `supplychain.SupplyChainEventImpacts` | Per-supplier impact of disruption events |

### Fixes applied in v2
- Split `SupplyChainEvents` into `SupplyChainEvents` (header) + `SupplyChainEventImpacts` (per-supplier impact)
- Populated `SupplierID` references in impact records (was previously empty)
- Removed denormalized columns (`SupplierName`, `ProductName`, etc.)
- Changed `ReliabilityScore` to `DECIMAL(5,2)` (was STRING)
- Added Primary Key (PK) and Foreign Key (FK) constraints on all tables

In [ ]:
spark.sql("CREATE SCHEMA IF NOT EXISTS supplychain")

In [ ]:
spark.sql("""
CREATE TABLE IF NOT EXISTS supplychain.Suppliers (
    SupplierID STRING NOT NULL,
    SupplierName STRING NOT NULL,
    SupplierType STRING,
    Status STRING,
    ProductLineName STRING,
    PrimarySupplierID STRING,
    LeadTimeDays INT,
    ReliabilityScore DECIMAL(5,2),
    Location STRING,
    ContactEmail STRING,
    CreatedBy STRING,
    CreatedDate DATE
) USING DELTA
""")

# spark.sql("ALTER TABLE supplychain.Suppliers ADD CONSTRAINT PK_Suppliers PRIMARY KEY (SupplierID)")

print("✅ supplychain.Suppliers created ")

In [ ]:
spark.sql("""
CREATE TABLE IF NOT EXISTS supplychain.ProductSuppliers (
    ProductSupplierID STRING NOT NULL,
    ProductID STRING NOT NULL,
    SupplierID STRING NOT NULL,
    SupplierProductCode STRING,
    WholesaleCost DECIMAL(10,2),
    MinOrderQuantity INT,
    MaxOrderQuantity INT,
    LeadTimeDays INT,
    Status STRING,
    CreatedBy STRING,
    CreatedDate DATE
) USING DELTA
""")

# spark.sql("ALTER TABLE supplychain.ProductSuppliers ADD CONSTRAINT PK_ProductSuppliers PRIMARY KEY (ProductSupplierID)")
# spark.sql("ALTER TABLE supplychain.ProductSuppliers ADD CONSTRAINT FK_PS_Product FOREIGN KEY (ProductID) REFERENCES product.Product(ProductID)")
# spark.sql("ALTER TABLE supplychain.ProductSuppliers ADD CONSTRAINT FK_PS_Supplier FOREIGN KEY (SupplierID) REFERENCES supplychain.Suppliers(SupplierID)")

print("✅ supplychain.ProductSuppliers created ")

In [ ]:
spark.sql("""
CREATE TABLE IF NOT EXISTS supplychain.SupplyChainEvents (
    EventID STRING NOT NULL,
    SupplierID INT,              -- FK to Suppliers.SupplierID (primary affected supplier)
    DisruptionType STRING,
    EventName STRING,
    Description STRING,
    Severity STRING,
    Status STRING,
    StartDate DATE,
    EndDate DATE,
    GeographicArea STRING,
    IndustryImpact STRING,
    PredictedDuration INT,
    ActualDuration INT,
    AlertLevel STRING,
    ReportedBy STRING,
    CreatedBy STRING,
    CreatedDate DATE
) USING DELTA
""")

# spark.sql("ALTER TABLE supplychain.SupplyChainEvents ADD CONSTRAINT PK_SupplyChainEvents PRIMARY KEY (EventID)")

print("✅ supplychain.SupplyChainEvents created ")

In [ ]:
spark.sql("""
CREATE TABLE IF NOT EXISTS supplychain.SupplyChainEventImpacts (
    ImpactID INT NOT NULL,
    EventID STRING NOT NULL,
    SupplierID STRING NOT NULL,
    ProductLineName STRING,
    ImpactLevel STRING,
    DeliveryDelay INT,
    CostIncrease DECIMAL(5,2),
    AlternativeAction STRING,
    EstimatedRevenueImpact DECIMAL(12,2),
    CreatedBy STRING,
    CreatedDate DATE
) USING DELTA
""")

# spark.sql("ALTER TABLE supplychain.SupplyChainEventImpacts ADD CONSTRAINT PK_EventImpacts PRIMARY KEY (ImpactID)")
# spark.sql("ALTER TABLE supplychain.SupplyChainEventImpacts ADD CONSTRAINT FK_Impact_Event FOREIGN KEY (EventID) REFERENCES supplychain.SupplyChainEvents(EventID)")
# spark.sql("ALTER TABLE supplychain.SupplyChainEventImpacts ADD CONSTRAINT FK_Impact_Supplier FOREIGN KEY (SupplierID) REFERENCES supplychain.Suppliers(SupplierID)")

print("✅ supplychain.SupplyChainEventImpacts created ")